# 04 · Statistical Significance & Publication Artifacts — *SycophancyRAGBench*

Computes paired McNemar tests with Benjamini-Hochberg FDR adjustments, Bootstrap 95% CIs, Cohen's Kappa, and generates publication-grade 300 DPI figures and LaTeX tables.

In [ ]:
import sys, os
pkg_root = os.path.abspath("..") if os.path.exists("../configs") else os.path.abspath(".")
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

from configs import config as cfg
from src import env, utils as U
import pandas as pd
from src.analysis.plots import generate_all_publication_artifacts
from src.analysis.stats import paired_mcnemar_test, apply_fdr_correction, compute_all_bootstrap_cis, compute_cohen_kappas
from IPython.display import Image, display

env.setup_workspace()
judged = U.read_jsonl(cfg.JUDGED_PATH)
df = pd.DataFrame(judged)
print(f"Loaded {len(df)} judged records.")

# Compute McNemar significance
models = sorted(df['model'].unique())
settings = sorted(df['setting'].unique())
mcnemar_rows = []
for m in models:
    for s in settings:
        if s != "vanilla":
            mcnemar_rows.append(paired_mcnemar_test(df, m, "j_endorses", "vanilla", s, belief_only=True))

mcnemar_rows = apply_fdr_correction(mcnemar_rows)
generate_all_publication_artifacts(df, mcnemar_rows)

fig1 = os.path.join(cfg.FIG_DIR, "sycophancy_by_model_and_retriever.png")
if os.path.exists(fig1):
    display(Image(fig1))
